In [3]:
import os
from pathlib import Path
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from chromadb import PersistentClient
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv
from groq import Groq
from pydantic import BaseModel, Field

In [ ]:
class Chunk(BaseModel):
    page_content: str
    metadata: dict

In [5]:
load_dotenv(override=True)

# openai = OpenAI()
# EMBEDDING_MODEL = "text-embedding-3-small"

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
chroma = PersistentClient(path="./chroma_db")
try:
    chroma.delete_collection("transcripts")
except:
    pass
collection = chroma.get_or_create_collection("transcripts")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
def load_documents(base_path="transcripts"):
    """
    Loops through the folders from the base_path and reads the txt files inside them
    Reads each transcript and stores text + metadata in a list of dicts
    Returns one dict(documents) per file with text, week, day, source
    """
    documents = [] #List of dicts
    base = Path(base_path) #Converts base_path into a Path object to use .glob and .iterdir
    

# Load documents from the transcripts folder
# If its not a folder, continue
# If it is a folder, iterate through all files in the folder
    for week_folder in sorted(base.iterdir()):
        if not week_folder.is_dir():
            continue
    # Iterate through all files in the week folder using .glob
        for file in sorted(week_folder.glob("*.txt")):
            text = file.read_text(encoding="utf-8")
            # Create a dictionary for each document and add it to the list
            documents.append({
                "text": text, # text of the transcript
                "week": week_folder.name, # name of the week folder
                "day": file.stem, # name of the file
                "source": str(file) # path of the file. file is a Path object and its converted to a string
            })
            print(f"Loaded: {file} ({len(text)} characters)")

    print(f"\nTotal documents loaded: {len(documents)}")
    return documents

documents = load_documents() #documents is the result of load_documents()

Loaded: transcripts/week1/day1.txt (26223 characters)
Loaded: transcripts/week1/day2.txt (58501 characters)
Loaded: transcripts/week1/day3.txt (60638 characters)
Loaded: transcripts/week1/day4.txt (63771 characters)
Loaded: transcripts/week1/day5.txt (49341 characters)
Loaded: transcripts/week2/day1.txt (56984 characters)
Loaded: transcripts/week2/day2.txt (48278 characters)
Loaded: transcripts/week2/day3.txt (24669 characters)
Loaded: transcripts/week2/day4.txt (43010 characters)
Loaded: transcripts/week2/day5.txt (41925 characters)
Loaded: transcripts/week3/day1.txt (57773 characters)
Loaded: transcripts/week3/day2.txt (37510 characters)
Loaded: transcripts/week3/day3.txt (31971 characters)
Loaded: transcripts/week3/day4.txt (49713 characters)
Loaded: transcripts/week3/day5.txt (34825 characters)
Loaded: transcripts/week4/day1.txt (40530 characters)
Loaded: transcripts/week4/day2.txt (41132 characters)
Loaded: transcripts/week4/day3.txt (37400 characters)
Loaded: transcripts/week4/da

In [7]:
def chunk_documents(documents):
    """
    Creates chunks and converts them into pydantic objects with metadata
    """
    # Creates the chunks using Langchains RecursiveCharacterTextSplitter
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        separators=["--- Lecture", "\n\n", "\n", " "]
    ) 
     # Loops on the documents list which is a list of dicts from the cell above  
    chunks = []
    for doc in documents:
        pieces = splitter.split_text(doc["text"]) # split_text splits documents into chunks based on the parameters from RecursiveCharacterTextSplitter.
        # Puts in pydantic structure for structured outputs and dot notation access
        for piece in pieces:
            chunks.append(Chunk(
                page_content=piece, # full text of the chunk
                metadata={"week": doc["week"], "day": doc["day"], "source": doc["source"]}
            ))
    print(f"Total chunks created: {len(chunks)}")
    print(f"\n--- Sample chunk ---\n")
    print(chunks[0].page_content)
    return chunks # A list of Chunk pydantic objects that include text, week, day, and source

In [8]:
def embed_and_store(chunks):
    """
    Create page_content, metadata, and ids and stores it in variables
    Encodes the texts into vectors
    Adds the vector into the chroma database
    """
    texts = [chunk.page_content for chunk in chunks] # A list of all the text in chunks using dot notation
    metadatas = [chunk.metadata for chunk in chunks] # Creates metadata for the week, day, and the source using dot notation
    ids = [f"{chunk.metadata['source']}_{i}" for i, chunk in enumerate(chunks)] # Names each chunk with the file name and an index

    # Turns the chunks into vectors through .encode
    embeddings = embedder.encode(texts, show_progress_bar=True).tolist()
    
    # Adds the vectors into the chroma database with the following data
    collection.add(
        documents=texts,
        embeddings=embeddings,
        metadatas=metadatas,
        ids=ids
    )
    print(f"Stored {len(chunks)} chunks in Chroma.")

chunks = chunk_documents(documents)
embed_and_store(chunks)

Total chunks created: 4287

--- Sample chunk ---

--- Lecture 1 ---
Okay, so now you scroll to this first box right here which says imports.
And the way that you run this bit of code known as a cell is you hold down the shift button and you
press return.
While you've clicked anywhere in this box, I press this and it runs.
Now, if you have any errors running this, if you get an import error or something like that, or if
it just doesn't seem to run, it just sits there.
It means your kernel isn't set right.
Look on the top right there.


Batches:   0%|          | 0/134 [00:00<?, ?it/s]

Stored 4287 chunks in Chroma.


In [9]:
# client = OpenAI()
client = Groq()
model = "openai/gpt-oss-120b"

def generate_answer(query, chunks, history=[]):
    context = "\n\n".join(chunk.page_content for chunk in chunks)
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": f"""You are a helpful study assistant for an LLM engineering course. Answer the user's question using only the context provided. If the answer isn't in the context, say so. Context: {context}"""},
      ] + history + [
          {"role": "user", "content": query}
        ]
    )   
    return response.choices[0].message.content

In [10]:
def rewrite_query(query, history=[]):
    """
    Calls the LLM to rewrite the query in a more clear and concise way
    """
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": f"""You are a search query optimizer for a knowledge base of LLM engineering course transcripts.
            Rewrite the user's question into a short, precise search query most likely to surface relevant content.
            
            This is the conversation history so far: {history}
            
            Respond ONLY with the rewritten query, nothing else."""},
            {"role": "user", "content": query}
        ]
    )   
    return response.choices[0].message.content

rewritten = rewrite_query("Whats there to love about rag and whats not to love?")
print(f"Original: Whats there to love about rag and whats not to love?")
print(f"Rewritten: {rewritten}")

Original: Whats there to love about rag and whats not to love?
Rewritten: pros and cons of Retrieval Augmented Generation (RAG)


In [11]:
def merge_chunks(chunks1, chunks2):
    merged = chunks1[:] # Everything from chunks1 which is a list of Chunk object
    existing = [chunk.page_content for chunk in chunks1] # Put the page_content of each object into a list
    
    # Checks if any chunk from chunk2 exists within the existing chunks
    for chunk in chunks2:
        if chunk.page_content not in existing:
            merged.append(chunk)
    return merged

In [54]:
def rerank(query, chunks):
    user_prompt = f"The user has asked the following question:\n\n{query}\n\nRank all chunks by relevance, most relevant first.\n\n"
    
    # Enumerates the chunk with an ID and the page_content to prepare the LLM to choose
    for i, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {i + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply with ONLY the chunk IDs as comma-separated integers, most relevant first. Example: 3,1,4,2,5..."

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a document re-ranker. Given a question and a list of chunks, return them ranked by relevance to the question, most relevant first. Respond in JSON format."},
            {"role": "user", "content": user_prompt}
        ],
    )
    order_str = response.choices[0].message.content.strip()
    order = [int(x.strip()) for x in order_str.split(',')] # Validates if order is a proper pydantic object and extracts the order list to a variable
    print(f"Total chunks sent to reranker: {len(chunks)}")
    print(f"Order returned by LLM: {order}")
    return [chunks[i - 1] for i in order] # Reorders chunks by using the ranked IDs from the LLM, converting from 1-indexed to 0-indexed

In [47]:
def fetch_context(query, n_results=20, history=[], final_k=10):
      query_embedding = embedder.encode(query).tolist() # Convert the query into a vector and puts it in a list
      results = collection.query(query_embeddings=[query_embedding], n_results=n_results) # Uses the list of vectors from query_embedding and gives n_results of similar vectors
      chunks1 = [Chunk(page_content=doc, metadata=meta) for doc, meta in zip(results["documents"][0], results["metadatas"][0])] # Converts the chunk's page_content and metadata which was in a list, back into a Chunk object

      rewritten = rewrite_query(query, history) # Rewrites the query into a concise way
      rewritten_embedding = embedder.encode(rewritten).tolist() # Converts the rewritten query into a vector and puts it in a list
      results2 = collection.query(query_embeddings=[rewritten_embedding], n_results=n_results) # Uses the list of vectors from rewritten_embedding and gives n_results of similar vectors
      chunks2 = [Chunk(page_content=doc, metadata=meta) for doc, meta in zip(results2["documents"][0], results2["metadatas"][0])] # Converts the chunk's page_content and metadata which was in a list, back into a Chunk object

      merged = merge_chunks(chunks1, chunks2) # Eliminates duplicates from the chunks
      reranked = rerank(query, merged) # Ranks them in order from best to worst
      return reranked[:final_k] # Gives the final_k best chunks

In [55]:
query = "Whats there to love about rag and whats not to love?"

rewritten = rewrite_query(query)
print(f"Rewritten: {rewritten}\n")

chunks = fetch_context(query)

for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} | {chunk.metadata['source']} ---")
    print(chunk.page_content)
    print()

answer = generate_answer(query, chunks)
print("=== Answer ===")
print(answer)

Rewritten: pros and cons of Retrieval Augmented Generation (RAG)

Total chunks sent to reranker: 36
Order returned by LLM: [1, 5, 9, 3, 2, 14, 18]
--- Chunk 1 | transcripts/week5/day4.txt ---
We just select the part that matters.
And then a very similar point is that it avoids us polluting the context with lots of information that's
irrelevant, not needed.
To answer this question, we can focus the context window on the most relevant information for the LLM
to answer this exact question.
Now I know what you're thinking.
You're thinking, this all sounds great.
What's there not to love about rag?
It's amazing.
And everyone people love rag.

--- Chunk 2 | transcripts/week5/day3.txt ---
And we're going to really double down on this tomorrow.
But but it's it's important to recognize that Rag is this very experimental, really quite hacky approach
of trying to look up relevant context.
And it can frequently go wrong.
You can get good examples of rag working really, really well.
Like it's surfa